# E2E baseline — Qwen2.5-7B-Instruct (4-bit) on a T4

Week-1 milestone: n=20 E2E-cleaned validation examples through the frozen LLM, then automatic metrics.

**Before running:** Runtime → Change runtime type → **T4 GPU**, and set `REPO_URL` below.

In [ ]:
REPO_URL = ""  # TODO: e.g. "https://github.com/<you>/hallucinations.git" (push the repo first)

import os
assert REPO_URL, "Set REPO_URL to your GitHub repository"
repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.exists(repo_name):
    !git clone {REPO_URL}
%cd {repo_name}

In [ ]:
%pip install -q -e ".[gpu]"

Optional: mount Google Drive and mirror `results/` there so runs survive session resets. Generation is resumable — rerunning skips cached ids.

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/t2t-results
    !rm -rf results && ln -s /content/drive/MyDrive/t2t-results results

In [ ]:
!python experiments/run_generation.py --config configs/e2e_baseline_colab20.yaml

In [ ]:
!python experiments/run_eval.py --config configs/e2e_baseline_colab20.yaml

In [ ]:
import json
from t2t.config import load_config, run_dir

rd = run_dir(load_config("configs/e2e_baseline_colab20.yaml"))
with open(rd / "generations.jsonl") as f:
    records = [json.loads(l) for l in f if l.strip()]
for rec in records[:5]:
    print(rec["id"], "→", rec["output"], "\n")